# 📊 Notebook 1: Data Exploration & Knowledge Base Construction**AI-Driven Insurance Chatbot — Diploma Project**This notebook explores the real Egyptian insurance dispensing rules dataset,cleans and structures the data, and builds a knowledge base for the AI chatbot.---

## 1. Setup & InstallationRun this cell first if you're on Google Colab:

In [ ]:
# Install required packages (Colab-ready)!pip install openpyxl pandas plotly -qimport pandas as pdimport jsonimport reimport openpyxlimport plotly.express as pximport plotly.graph_objects as gofrom collections import Counterprint('All packages loaded successfully!')

## 2. Load the DatasetUpload the Excel file `نظم صرف شركات التأمين-2.xlsx` to Colab,or modify the path below to point to your local file.

In [ ]:
# For Google Colab: upload the file# from google.colab import files# uploaded = files.upload()# Load the Excel fileEXCEL_PATH = 'نظم صرف شركات التأمين-2.xlsx'  # Update path if neededwb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)ws = wb['Sheet1']print(f'Sheet loaded: {ws.max_row} rows x {ws.max_column} columns')# Show column headersheaders = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]print(f'Columns: {headers}')

## 3. Convert to DataFrame for Exploration

In [ ]:
# Read all rows into a DataFramedata = []for r in range(2, ws.max_row + 1):    row = {        'company_code': str(ws.cell(r, 1).value or '').strip(),        'company_name': str(ws.cell(r, 2).value or '').strip(),        'category': str(ws.cell(r, 3).value or '').strip(),        'details': str(ws.cell(r, 4).value or '').strip(),        'notes': str(ws.cell(r, 5).value or '').strip(),    }    data.append(row)df = pd.DataFrame(data)print(f'Total rows: {len(df)}')print(f'Unique companies: {df["company_name"].nunique()}')print(f'Unique categories: {df["category"].nunique()}')df.head(10)

## 4. Exploratory Analysis### 4.1 Insurance Companies Overview

In [ ]:
# Count of policy categories per companycompany_cats = df.groupby('company_name')['category'].nunique().reset_index()company_cats.columns = ['Company', 'Number of Categories']company_cats = company_cats.sort_values('Number of Categories', ascending=False)fig = px.bar(company_cats.head(20), x='Number of Categories', y='Company',             orientation='h', title='Top 20 Companies by Number of Policy Categories',             color='Number of Categories',             color_continuous_scale=['#1a1f2e', '#00D4AA'])fig.update_layout(height=600)fig.show()print(f'\nCompanies with most categories: {company_cats.iloc[0]["Company"]} ({company_cats.iloc[0]["Number of Categories"]})')print(f'Companies with fewest categories: {company_cats.iloc[-1]["Company"]} ({company_cats.iloc[-1]["Number of Categories"]})')

### 4.2 Category Distribution

In [ ]:
# How many companies have each category?cat_coverage = df.groupby('category')['company_name'].nunique().reset_index()cat_coverage.columns = ['Category', 'Companies']cat_coverage = cat_coverage.sort_values('Companies', ascending=True)fig2 = px.bar(cat_coverage, x='Companies', y='Category', orientation='h',              title='Category Coverage: How Many Companies Have Each Rule Type',              color='Companies',              color_continuous_scale=['#ef4444', '#f59e0b', '#10b981'])fig2.update_layout(height=500)fig2.show()

### 4.3 Exclusion (المحظورات) Analysis

In [ ]:
# Extract and analyze exclusion listsexclusions = df[df['category'] == 'المحظورات']print(f'Companies with exclusion lists: {len(exclusions)} out of {df["company_name"].nunique()}')# Common excluded items across companiescommon_terms = ['تجميل', 'فيتامين', 'حمل', 'تخسيس', 'منشطات', 'عدسات', 'أسنان', 'شعر']term_counts = {}for _, row in exclusions.iterrows():    details_lower = row['details'].lower()    for term in common_terms:        if term in details_lower:            term_counts[term] = term_counts.get(term, 0) + 1term_df = pd.DataFrame(sorted(term_counts.items(), key=lambda x: x[1], reverse=True),                       columns=['Excluded Category', 'Companies'])fig3 = px.bar(term_df, x='Companies', y='Excluded Category', orientation='h',              title='Most Common Exclusion Types Across Companies',              color='Companies', color_continuous_scale=['#dc2626', '#f59e0b'])fig3.show()

## 5. Build the Knowledge BaseStructure the data into a JSON knowledge base for the chatbot engine.

In [ ]:
# Category translationsCATEGORY_TRANSLATIONS = {    'نماذج الصرف': 'Dispensing Forms',    'المحظورات': 'Excluded / Prohibited Items',    'التحمل': 'Co-payment / Deductible',    'التشخيص': 'Diagnosis Requirements',    'صلاحية النموذج': 'Form Validity Period',    'صورة البطاقة': 'National ID Copy Requirement',    'صورة الكارنية': 'Insurance Card Copy Requirement',    'الختم / إمضاء العميل': 'Stamp / Signature Requirements',    'أقصى مدة للصرف': 'Maximum Dispensing Duration',    'الحد الأقصى': 'Maximum Financial Limit',    'التواصل للموافقات': 'Approval Contact Information',    'لينك الاونلاين سيستم': 'Online System Link',    'البدائل': 'Generic Substitution Rules',    'ملاحظات': 'Additional Notes',}# Build knowledge baseknowledge_base = {}for _, row in df.iterrows():    company = row['company_name']    if company not in knowledge_base:        knowledge_base[company] = {            'company_name': company,            'policies': {}        }    cat = row['category']    knowledge_base[company]['policies'][cat] = {        'category_ar': cat,        'category_en': CATEGORY_TRANSLATIONS.get(cat, cat),        'details': row['details'],        'notes': row['notes'],    }print(f'Knowledge base built: {len(knowledge_base)} companies')total_policies = sum(len(c["policies"]) for c in knowledge_base.values())print(f'Total policy entries: {total_policies}')

In [ ]:
# Save to JSONwith open('insurance_knowledge_base.json', 'w', encoding='utf-8') as f:    json.dump(knowledge_base, f, ensure_ascii=False, indent=2)print('Knowledge base saved to insurance_knowledge_base.json')

## 6. Summary| Metric | Value ||--------|-------|| Total Insurance Companies | 77 || Total Policy Entries | 766 || Unique Rule Categories | 14 || Companies with Exclusion Lists | ~50+ || Data Source | Real Egyptian Insurance Rules |**Next:** Notebook 2 will build the NLP chatbot engine using TF-IDF.